<a href="https://colab.research.google.com/github/kechsin/childes_ner/blob/main/main_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Транскрипция whisper

In [ ]:
num = "00028"
vname = "00028.MTS"
auname = "audio00028.wav"
#auname_norm = "audio00024_N.wav"

#надо ещё поменять в ячейке !ffmpeg

In [ ]:
!pip3 install whisper-timestamped

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.0/825.0 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=c9c8cf861098e23e09d61c00e92976bafa508d17237710173b6504a6537498b2
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^

In [ ]:
import os
import numpy as np

try:
    import tensorflow  # required in Colab to avoid protobuf compatibility issues
except ImportError:
    pass

import torch
import pandas as pd
import whisper
import whisper_timestamped
import torchaudio
#import ffmpeg

from tqdm.notebook import tqdm
import json

Exception ignored in: <function _xla_gc_callback at 0x7a0fb42622a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


ModuleNotFoundError: No module named 'whisper'

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
option = 'Apel-sin'
#option = 'antony66'

In [ ]:
if option == 'Apel-sin':
    model = whisper_timestamped.load_model("Apel-sin/whisper-large-v3-russian-ties-podlodka-v1.2", device=DEVICE)
else:
    model = whisper_timestamped.load_model("antony66/whisper-large-v3-russian", device=DEVICE)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [ ]:
audio = whisper_timestamped.load_audio(auname)
transcript = whisper_timestamped.transcribe(model, audio, language="ru", beam_size=5, best_of=5, temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),)

100%|██████████| 102103/102103 [16:15<00:00, 104.65frames/s]


In [ ]:
with open(f"transcript_{num}_{option}.json", "w") as f:
    f.write(json.dumps(transcript, indent = 2, ensure_ascii = False))

In [ ]:
text = transcript['text']

# Сравнение транскрипций

In [ ]:
!pip install --upgrade evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.1 MB/s eta 0:00:00


In [ ]:
import re

def only_letters(dict_transcript):
    only_letters = ""
    for seg in dict_transcript['segments']:
        for word in seg['words']:
            add_word = word['text'].lower()
            add_word.replace('-', ' ')
            only_letters += re.sub(r"[^a-zA-Zа-яА-ЯеЁ]", "", add_word) + " "

    return only_letters

In [ ]:
import json
for num in ['00024', '00028sh']:
    transcripts = {'golden': None, 'antony66': None, 'Apel-sin': None,
                   'ctc': None, 'e2e_rnnt': None, 'e2e_ctc': None}
    for i in transcripts:
        with open(f"transcript_{num}_{i}.json") as f:
            print(f"transcript_{num}_{i}.json")
            transcripts[i] = only_letters(json.load(f))

transcript_00024_golden.json
transcript_00024_antony66.json
transcript_00024_Apel-sin.json
transcript_00024_ctc.json
transcript_00024_e2e_rnnt.json
transcript_00024_e2e_ctc.json
transcript_00028sh_golden.json
transcript_00028sh_antony66.json
transcript_00028sh_Apel-sin.json
transcript_00028sh_ctc.json
transcript_00028sh_e2e_rnnt.json
transcript_00028sh_e2e_ctc.json


In [ ]:
from evaluate import load

wer_metric = load("wer")

In [ ]:
wer_losses = {'antony66': None, 'Apel-sin': None,
                   'ctc': None, 'e2e_rnnt': None, 'e2e_ctc': None}
for tr in wer_losses:
    wer_losses[tr] = wer_metric.compute(references=[transcripts['golden']], predictions=[transcripts[tr]])

In [ ]:
wer_losses

{'antony66': 0.35008665511265163,
 'Apel-sin': 0.7712305025996534,
 'ctc': 0.19064124783362218,
 'e2e_rnnt': 0.24783362218370883,
 'e2e_ctc': 0.24436741767764297}

In [ ]:
cer_metric = load("cer")
cer_losses = {'antony66': None, 'Apel-sin': None,
                   'ctc': None, 'e2e_rnnt': None, 'e2e_ctc': None}
for tr in cer_losses:
    cer_losses[tr] = cer_metric.compute(references=[transcripts['golden']], predictions=[transcripts[tr]])

In [ ]:
cer_losses

{'antony66': 0.23940543959519292,
 'Apel-sin': 0.7179000632511069,
 'ctc': 0.1312460468058191,
 'e2e_rnnt': 0.18121442125237192,
 'e2e_ctc': 0.14389626818469323}

# Утилиты

In [ ]:
import json

def read_file(num):
    fname = f"transcript{num}.json"
    with open(fname) as f:
        transcript = json.load(f)
    return transcript

In [ ]:
def concat_lists(list1, list2):
    heading = [('Time Stamp', 'Proper Nouns'), ('', '')]
    list1 = list1[2:]
    list2 = list2[2:]
    lists = sorted(set(list1 + list2))
    return heading + lists

In [ ]:
def index_timestamp(transcript, start, end):
    #print(transcript['text'][35:40])
    segments = transcript['segments']
    len_segments = 0
    answer = []
    for segment_number, i in enumerate(segments):
        if not 'text' in i or len_segments + len(i['text']) > start:
            len_words = 0
            for word_number, j in enumerate(i['words']):
                #print(len_words)
                if len_words >= start - len_segments and len_words < end - len_segments:
                    answer.append({'seg_num': segment_number,
                          'word_num': word_number,
                          'text': j['text'],
                          'start': j['start'],
                          'end': j['end']})
                if len_words >= end - len_segments:
                    return answer
                #print('spaces' + str(1 + int(transcript['text'][len_segments + len_words + 1] == ' ')))
                #print('len' + str(len(j['text'].strip())))
                len_words += len(j['text'].strip()) + 1

        if 'text' in i:
            len_segments += len(i['text'])
    return answer

In [ ]:
def delete_exceptions(tstamps):
    exceptions = ['алиса', 'алисе', 'алису', 'алисой', 'алисы']
    deleting = []
    for i, line in enumerate(tstamps):
        for j in exceptions:
            if line[1].strip(' .?,"!?\\/-').lower() == j:
                deleting.append(i)
    for i in deleting[::-1]:
        tstamps.pop(i)
    return tstamps

# NER 0 (по списку)

In [ ]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 57.3 MB/s eta 0:00:00


In [ ]:
import pymorphy3
import os

def save_forms():
    morph = pymorphy3.MorphAnalyzer()

    with open('names.txt') as f:
        names = f.readlines()
        names = [i.strip() for i in names]

    forms = set()
    for i in names:
        if i.find(':') == -1:
            continue
        name, category = i.split(':')
        forms.add(name.lower() + '\n')
        if category == 'nom':
            if name[-1] == 'а':
                forms.add((name[:-1] + '\n').lower())
            elif name[-1] == 'я':
                forms.add((name[:-1] + 'ь' + '\n').lower())
            for option in morph.parse(name):
                if 'Name' in option.tag:
                    for morph_case in ('gent', 'accs', 'datv', 'ablt', 'loct'):
                        form = option.inflect({morph_case})
                        forms.add(form.word + '\n')
        if category == 'poss':
            for option in morph.parse(name):
                if 'ADJF' in option.tag:
                    for morph_case in ('gent', 'accs', 'datv', 'ablt', 'loct'):
                        for gender_number in ('masc', 'femn', 'neut', 'plur'):
                            form = option.inflect({morph_case, gender_number})
                            forms.add(form.word + '\n')
    with open('forms.txt', 'w') as f:
        f.writelines(sorted(forms))



In [ ]:
def list_based(transcript):
    if not os.path.exists('forms.txt'):
        if not os.path.exists('names.txt'):
            print('Вам нужно составить список names.txt. Правила будут где-нибудь описаны и будет пример.')
        else:
            save_forms()
    with open('forms.txt') as f:
        forms = {i.strip() for i in f.readlines()}
    tstamps0 = [('Time Stamp', 'Proper Nouns'), ('', '')]
    punctuation = '.,?!":;-—'
    for seg in transcript['segments']:
        for word in seg['words']:
            w_text = word['text'].lower()
            for sign in punctuation:
                w_text = w_text.replace(sign, '')
            if w_text in forms:
                tstamps0.append((word['start'], word['text']))
    return tstamps0


#NER 1
roberta-ner


In [ ]:
!pip install -q transformers

from transformers import pipeline
import copy
import csv

In [ ]:
def roberta_single(transcript, ner):
    def join_entities(entities):
        entities2 = copy.deepcopy(entities)
        for n, ent in enumerate(entities[::-1]):
            if ent['entity'][0] == "I":
                index = len(entities) - n - 1
                entities2[index-1]['word'] = entities[index-1]['word'] + entities2[index]['word']
                entities2[index-1]['end'] = entities2[index]['end']
                del entities2[index]
        return entities2


    text = transcript['text']

    entity = ner(text)
    print(entity)
    entities = join_entities(entity)
    tstamps = [('Time Stamp', 'Proper Nouns'), ('', '')]
    for i in entities:
        results = index_timestamp(transcript, i['start'], i['end'])
        for result in results:
            tstamps.append((result['start'], result['text']))
    print(entities)
    return tstamps

In [ ]:
def roberta_ner(transcripts):
    ner = pipeline("ner", model="yqelz/xml-roberta-large-ner-russian")
    tstamp_list = []
    file_number = 0
    for transcript in transcripts:
        print(f'Обрабатываем файл номер {file_number}')
        tstamps = roberta_single(transcript, ner)
        tstamp_list.append(tstamps)
        file_number += 1
    return tstamp_list

# NER 2
SpaCy

In [ ]:
!pip install spacy
!python -m spacy download ru_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 19.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

def spacy_single(transcript, nlp):
    text = transcript['text']
    doc = nlp(text)
    tstamps = [('Time Stamp', 'Proper Nouns'), ('', '')]
    for ent in doc.ents:
        # print(ent.text, ent.start_char, ent.end_char, ent.label_)
        results = index_timestamp(transcript, ent.start_char, ent.end_char)
        for result in results:
            tstamps.append((result['start'], result['text']))
    return tstamps


def spacy_ner(transcripts):
    nlp = spacy.load("ru_core_news_md")
    tstamp_list = []
    file_number = 0
    for transcript in transcripts:
        print(f'Обрабатываем файл номер {file_number}')
        tstamps = spacy_single(transcript, nlp)
        tstamp_list.append(tstamps)
        file_number += 1
    return tstamp_list

# NER 3
Natasha

In [ ]:
!pip install slovnet
!pip install navec
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.0 MB/s eta 0:00:00
--2026-05-28 11:09:48--  https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634240 (25M) [application/x-tar]
Saving to: ‘navec_news_v1_1B_250K_300d_100q.tar’

navec_news_v1_1B_25 100%[===================>]  25.40M  11.1MB/s    in 2.3s    

2026-05-28 11:09:51 (11.1 MB/s) - ‘navec_news_v1_1B_250K_300d_100q.tar’ saved [26634240/26634240]



In [ ]:
from navec import Navec
from slovnet import NER

def natasha_single(transcript, ner):
    text = transcript['text']
    markup = ner(text)
    tstamps = [('Time Stamp', 'Proper Nouns'), ('', '')]
    for ent in markup.spans:
        # print(ent.text, ent.start_char, ent.end_char, ent.label_)
        results = index_timestamp(transcript, ent.start, ent.stop)
        for result in results:
            tstamps.append((result['start'], result['text']))
        # print(ent.start, ent.stop)
    return tstamps


def natasha_ner(transcripts):
    navec = Navec.load('navec_news_v1_1B_250K_300d_100q.tar')
    ner = NER.load('slovnet_ner_news_v1.tar')
    ner.navec(navec)
    tstamp_list = []
    file_number = 0
    for transcript in transcripts:
        print(f'Обрабатываем файл номер {file_number}')
        tstamps = natasha_single(transcript, ner)
        tstamp_list.append(tstamps)
        file_number += 1
    return tstamp_list

#Пайплайн

Опции:


roberta_ner

spacy_ner

natasha_ner

In [ ]:
import re
import csv
def process_transcripts(names, ner_function, testing=True):
    transcripts = []
    for name in names:
        transcripts.append(read_file(name))
    for i in range(len(transcripts)):
        transcripts[i]['text'] = re.sub(r'\s+', ' ', transcripts[i]['text']).strip()
    timestamps1 = ner_function(transcripts)
    timestamps0 = [list_based(transcript) for transcript in transcripts]

    if testing:
        funname = str(ner_function).split()[1] + '_'
    else:
        funname = ""
    for i in range(len(names)):
        all_timestamps = concat_lists(timestamps0[i], timestamps1[i])
        all_timestamps = delete_exceptions(all_timestamps)
        with open(f'{funname}{names[i]}.csv', 'w') as f:
            writer = csv.writer(f)
            writer.writerows(all_timestamps)

In [ ]:
numbers = ['00024', '00028', '00059', '00079', '00038', '00036']

names = list([f'_{i}_e2e_ctc' for i in numbers])
process_transcripts(names, roberta_ner)
process_transcripts(names, spacy_ner)
process_transcripts(names, natasha_ner)

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: yqelz/xml-roberta-large-ner-russian
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Обрабатываем файл номер 0
[{'entity': 'B-ORG', 'score': np.float32(0.722127), 'index': 6, 'word': '▁юбилей', 'start': 15, 'end': 21}, {'entity': 'I-ORG', 'score': np.float32(0.7254941), 'index': 7, 'word': 'ной', 'start': 21, 'end': 24}, {'entity': 'I-ORG', 'score': np.float32(0.51702785), 'index': 8, 'word': '▁улице', 'start': 25, 'end': 30}]
[{'entity': 'B-ORG', 'score': np.float32(0.722127), 'index': 6, 'word': '▁юбилейной▁улице', 'start': 15, 'end': 30}]
Обрабатываем файл номер 1
[{'entity': 'I-PER', 'score': np.float32(0.62474257), 'index': 22, 'word': 'ша', 'start': 50, 'end': 52}, {'entity': 'I-PER', 'score': np.float32(0.61021), 'index': 32, 'word': 'ш', 'start': 83, 'end': 84}, {'entity': 'B-PER', 'score': np.float32(0.6729188), 'index': 40, 'word': '▁Пе', 'start': 107, 'end': 109}, {'entity': 'I-PER', 'score': np.float32(0.5261648), 'index': 41, 'word': 'б', 'start': 109, 'end': 110}, {'entity': 'B-PER', 'score': np.float32(0.5014091), 'index': 44, 'word': '▁Пе', 'start': 113

# Сравнение NER

In [ ]:
!pip install levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.3 MB/s eta 0:00:00


In [ ]:
import Levenshtein
import csv
def similar(word1, word2, time_dif=0.05, ratio=0.1):
    if abs(word1[0] - word2[0]) > time_dif:
        return False
    striped_w1 = word1[1].strip(' .?,"!?\\/-').lower()
    striped_w2 = word2[1].strip(' .?,"!?\\/-').lower()
    # по дефолту можно разницу в 1 символ на каждые 5 букв одного из слов
    return Levenshtein.ratio(striped_w1, striped_w2) >= 1 - ratio


def compare_lists(golden_ner, other_ner):
    tp = fp = fn = 0
    i = j = 2
    while i < len(golden_ner) and j < len(other_ner):
        if golden_ner[i][0] == other_ner[j][0] or similar(golden_ner[i], other_ner[j]):
            tp += 1
            #print('tp', other_ner[j])
            i += 1
            j += 1
        elif golden_ner[i][0] < other_ner[j][0]:
            fn += 1
            #print('fn', golden_ner[i])
            i += 1
        else:
            fp += 1
            #print('fp', other_ner[j])
            j += 1
    fp += len(other_ner) - j
    fn += len(golden_ner) - i
    recall = tp/(tp+fn)
    precision = tp/(tp+fp)

    return {'tp': tp, 'fn': fn, 'fp': fp,
            'recall': recall, 'precision': precision,
            'F1': 2*recall*precision/(recall + precision)}


In [ ]:
import json
def metrics(num, transcript_option = 'e2e_ctc'):
    ner_lists = {'golden': [], 'roberta': [], 'spacy': [], 'natasha': []}
    for option in ner_lists:
        with open(f'{option}_ner__{num}_{transcript_option}.csv', newline='') as f:
            reader = csv.reader(f)
            for row in reader:
                if row[0] == 'Time Stamp' or row[0] == '':
                    ner_lists[option].append(row)
                else:
                    ner_lists[option].append([float(row[0]), row[1]])
    results = {'roberta': None, 'spacy': None, 'natasha': None}
    for option in results:
        results[option] = compare_lists(ner_lists['golden'], ner_lists[option])
    with open(f'metrics_{num}.json', 'w') as f:
        json.dump(results, f)
    print('Метрики сохранены')

In [ ]:
def join_metrics(metrics_list):
    result = {'roberta': None, 'natasha': None, 'spacy': None}
    for option in result:
        tp = sum([i[option]['tp'] for i in metrics_list])
        fp = sum([i[option]['fp'] for i in metrics_list])
        fn = sum([i[option]['fn'] for i in metrics_list])
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        result[option] =  {'tp': tp, 'fn': fn, 'fp': fp,
                     'recall': recall, 'precision': precision,
                     'F1': 2*recall*precision/(recall + precision)}
    return result

In [ ]:
#for i in ['00024', '00028', '00038', '00059', '00079']:
for i in ['00024m']:
    metrics(i)

Метрики сохранены


In [ ]:
import json

metrics_list = []
for i in ['00024', '00028', '00059', '00079', '00038']:
    with open(f'metrics_{i}.json') as f:
        metrics_list.append(json.load(f))

In [ ]:
import json

metrics_list = []
for i in ['00028br', '00182shbr']:
    with open(f'metrics_{i}.json') as f:
        metrics_list.append(json.load(f))

In [ ]:
final_metrics = join_metrics(metrics_list)
with open('metrics_br.json', 'w') as f:
    json.dump(final_metrics, f)

In [ ]:
for i in ['00024', '00028', '00038', '00059', '00079']:
    transcript = read_file('_' + i + '_e2e_ctc')
    words = 0
    for i in transcript['segments']:
        words += len(i['words'])
    print(words)

790
1181
505
1380
1530
